# Reading evidence: packets, Linux, Windows and network sensors

Read four kinds of evidence from one night at a clinic: a Linux server's auth log and processes, a packet capture, Zeek connection logs and Windows event logs. Find a brute force, a script download, a beacon and a hop to a reception PC, then write a host triage note and a network finding.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `auth.log`, `capture.pcap`, `crontab.txt`, `eventlogs/Security.jsonl`, `ps.txt`, `README.txt`, `recovered/m.sh`, `zeek/conn.log`.

## 1. Take stock of the evidence

Start every case the same way: read the README, list what you were given, and hash it before you open anything. The hashes prove later that nobody changed the evidence after you received it.

- [ ] Read README.txt: which machines, which addresses, which times
- [ ] List every file
- [ ] Hash every file and keep the list

> Hint: `ls -R` lists folders inside folders. `sha256sum` prints one hash per file.

?> How many evidence files are there, including the README? :: sha256:2c624232cdd221771294dfbb310aca000a0df6ac8b66b696d90ef06fdefb64a3
?> What is the IP address of appt01? :: sha256:4b0ea0f668396f131ec196b64f8ec381999a084e3a47b8da2e5f755a65e184c4

In [ ]:
%%bash
cd ~/case
ls -R
echo ---
sha256sum auth.log ps.txt crontab.txt capture.pcap README.txt recovered/m.sh zeek/conn.log eventlogs/Security.jsonl

## 2. Linux: who was knocking

appt01 is open to the internet on port 22. `auth.log` records every SSH login, good or bad. Count the failures, then count them **by source address**: one address making dozens of guesses is a brute force.

- [ ] How many failed passwords in total?
- [ ] Which address made most of them, and how many?
- [ ] Which user names did it try? Which one is a real account here?

> Hint: `grep -o 'from [0-9.]*'` keeps only the address part of each line; `sort | uniq -c | sort -rn` counts and ranks them.

?> How many failed passwords are in auth.log in total? :: sha256:1a6562590ef19d1045d06c4055742d38288e9e6dcd71ccde5cee80f1d5a774eb
?> Which address made the most failed attempts? :: sha256:bfeb4c6192985efa05e7fa0740ac45708a515e569e7edaec7fc060ff72b44a0c
?> How many failed attempts came from that address? :: sha256:25fc0e7096fc653718202dc30b0c580b8ab87eac11a700cba03a7c021bc35b0c

In [ ]:
%%bash
cd ~/case
grep -c 'Failed password' auth.log
echo ---
grep 'Failed password' auth.log | grep -o 'from [0-9.]*' | sort | uniq -c | sort -rn
echo ---
grep 'Failed password' auth.log | grep -o 'for [a-z ]*from' | sort | uniq -c | sort -rn

## 3. Linux: the login and what came next

A brute force only matters if it worked. Find the **Accepted** line, then read everything the account did afterwards: `sudo` lines show each command run as root. Then check the two things an attacker leaves to come back: a scheduled job and a running process.

- [ ] Which account logged in from 198.51.100.23, when, and with what (password or key)?
- [ ] Which commands did it run with sudo?
- [ ] What did the script change? Read recovered/m.sh
- [ ] Which cron line and which process are new?

> Hint: The deploy account logs in with a key from the IT admin PC every night; that one is normal. Compare the cron lines with what the script added.

?> Which account logged in with a password from 198.51.100.23? :: sha256:630ba09448af522154f38ef7685ef1f44b0f3e9430f80829a03ce24f400f3754
?> What file does the new cron job run? (full path) :: sha256:b57fa7460b593d55238865605d1856f2f10920d596db667477e2b06fc4ca7e6a
?> At what time (UTC, HH:MM:SS) did that login succeed? :: sha256:dcd0265553c98b2647b88d9fc0ae73807c06ce0d9ca87c75e1934436a0dd87f9

In [ ]:
%%bash
cd ~/case
grep 'Accepted' auth.log
echo ---
grep 'sudo' auth.log | grep COMMAND
echo ---
cat recovered/m.sh
echo ---
cat crontab.txt
echo ---
grep -E 'sysmon|\.m\.sh' ps.txt

## 4. Packets: the download

Now the wire. The capture is the core switch mirror from 01:15. In the packet viewer a **display filter** keeps only the packets you ask for. Start with every HTTP request, then everything to or from the download server, and follow the stream of the first download.

```tool packets
{"file": "capture.pcap", "f": "http.request", "n": 2, "blank": ["f"]}
```

```tool packets
{"file": "capture.pcap", "f": "ip.addr==203.0.113.77", "n": 10, "blank": ["f"]}
```

The Linux cell below asks tshark the same question on the command line.

- [ ] Which two files were fetched, from which host name?
- [ ] Which program fetched each one? (the User-Agent)
- [ ] What kind of file came back each time?

> Hint: The first request's User-Agent is Wget, which matches the sudo line. The second is curl, which matches the script.

?> Which host name served /m.sh? :: sha256:817ce8dbe01257e2e21aca4761c28838ae142340ef0c0e5ea0c955264c1246e4
?> Which User-Agent fetched /a.bin? :: sha256:288d55347fe66e2a73b6a951f78d3729876d7bc80cf71082a5ef4addbf753f8b

In [ ]:
%%bash
cd ~/case
tshark -r capture.pcap -Y 'http.request' -T fields -e ip.src -e http.host -e http.request.uri -e http.user_agent

## 5. Packets: the beacon

After the download, appt01 keeps calling the same server over TLS. The content is encrypted, but the **timing** and the **server name** (SNI) are not. Regular calls at a fixed interval are a beacon: a program checking in for instructions.

```tool packets
{"file": "capture.pcap", "f": "tls and ip.dst==203.0.113.77", "n": 3, "blank": ["f"]}
```

```tool packets
{"file": "capture.pcap", "f": "dns.qry.name contains patch-relay", "n": 8, "blank": ["f"]}
```

- [ ] When does each TLS Client Hello to 203.0.113.77 start?
- [ ] How far apart are they? Does that match the cron line?
- [ ] Compare with the TLS to portal and backup: why are those normal?

> Hint: Look at the Time column. The cron line runs every ten minutes.

?> How many minutes apart are the TLS connections to 203.0.113.77? :: sha256:4a44dc15364204a80fe80e9039455cc1608281820fe2b24f1e5233ade6af1dd5

## 6. Network sensor: who talked to whom

Zeek writes one line per connection: who opened it, to which address and port, how long it lasted and how many bytes went each way. It's the fastest way to see the whole night at once. The fields are separated by tabs; the `#fields` line names them.

- [ ] List every pair: from, to, port, service, with a count
- [ ] How many connections did appt01 open to 203.0.113.77?
- [ ] Which internal machine did appt01 connect to on port 445 (Windows file sharing)? Why is that strange for an appointments server?
- [ ] Which connection moved the most bytes? Is it suspicious?

> Hint: Field 3 is id.orig_h (who opened it), 5 is id.resp_h, 6 is id.resp_p, 8 is service and 11 is resp_bytes. The biggest transfer is the nightly backup, which is internal and runs at 01:30 every night.

?> How many connections did appt01 open to 203.0.113.77? :: sha256:ef2d127de37b942baad06145e54b0c619a1f22327b2ebbcfbec78f5564afe39d
?> Which internal address did appt01 connect to on port 445? :: sha256:13dd5dc4e7aea0de7349cfd02701ce0836f6b9fbb32b4636f5086eda771a983a

In [ ]:
%%bash
cd ~/case/zeek
grep -v '^#' conn.log | awk -F'\t' '{print $3, "->", $5":"$6, $8}' | sort | uniq -c | sort -rn
echo ---
grep -v '^#' conn.log | awk -F'\t' '$5=="10.30.2.33" {print $1, $3, $6, $12, $11}'

## 7. Windows: the hop to reception

The reception PC saw those port 445 connections as **logons**. In the Security log, 4625 is a failed logon and 4624 a successful one; Logon Type 3 means over the network. 5140 is a share being opened and 5145 a file in it being read.

```tool events
{"file": "eventlogs/Security.jsonl", "ids": "4625", "q": "10.30.1.20", "n": 9, "blank": ["ids", "src", "q"]}
```

```tool events
{"file": "eventlogs/Security.jsonl", "ids": "4624, 5140, 5145", "q": "10.30.1.20", "n": 8, "blank": ["ids", "src", "q"]}
```

- [ ] How many failed logons came from appt01, for which account?
- [ ] When did the logon succeed, and with which Logon Type?
- [ ] Which share was opened, and which files were read?
- [ ] What kind of data is in that share? Why does that raise the stakes?

> Hint: The one 4625 on 5 Nov at 03:31 is the receptionist mistyping at the keyboard (Logon Type 2). It isn't part of this.

?> How many failed logons came from 10.30.1.20? :: sha256:19581e27de7ced00ff1ce50b2047e7a567c76b1cbaebabe5ef03f7c3017bb5b7
?> Which share did the attacker open on rcp-pc03? :: sha256:5982f892947ea523f3a9473a5af9ded238c1204127e1e595e845e7555ba6c99c
?> How many files were read from it? :: sha256:e7f6c011776e8db7cd330b54174fd76f7d0216b612387a5ffcfb81e6f0919683

## 8. A sensor rule for next time

Turn what you found into rules the IDS can run on every packet, so the next download and the next beacon raise an alert by themselves. Run them on the capture and check two things: they catch the bad traffic, and they don't fire on the portal, the backup or the database.

```tool ids
{"rules": "alert http $HOME_NET any -> $EXTERNAL_NET any (msg:\"Script or binary fetched by wget or curl\"; http.user_agent; pcre:\"/^(Wget|curl)\\//\"; sid:1100001; rev:1;)\nalert tls $HOME_NET any -> $EXTERNAL_NET 443 (msg:\"Beacon to cdn.patch-relay.example\"; content:\"SNI=cdn.patch-relay.example\"; sid:1100002; rev:1;)\n", "file": "capture.pcap", "n": 5, "blank": ["rules"]}
```

- [ ] Rule 1 catches both downloads
- [ ] Rule 2 catches the three beacons
- [ ] No alert on internal traffic
- [ ] One line on what each rule would miss

> Hint: A real server rarely needs Wget or curl to reach the internet, so those User-Agents are worth an alert there. A beacon rule on one server name is easy to evade with a new name; the timing is the more lasting sign.

?> How many alerts do the two rules give on the capture? :: sha256:ef2d127de37b942baad06145e54b0c619a1f22327b2ebbcfbec78f5564afe39d

## 9. Write the triage note and the network finding

Two short documents carry this to the next person: a **host triage note** for appt01 and a **network finding**. Each says what you checked, with the exact commands and filters, so anyone can repeat it. Keep facts apart from guesses.

```tool report
{"tpl": "triage", "title": "Triage note: appt01 compromised through SSH password guessing", "text": "# Triage note: appt01 compromised through SSH password guessing\n\n## What fired\n\nIDS alert at 01:20:11 UTC on 6 Nov: script download by Wget from appt01 (10.30.1.20) to 203.0.113.77 (cdn.patch-relay.example).\n\n## What I checked\n\n- auth.log: `grep 'Failed password' auth.log | grep -o 'from [0-9.]*' | sort | uniq -c | sort -rn`, then `grep Accepted` and the sudo COMMAND lines.\n- crontab.txt, ps.txt and recovered/m.sh.\n- Capture: filters `http.request`, `ip.addr==203.0.113.77`, `tls and ip.dst==203.0.113.77`.\n- Zeek conn.log pairs, and rcp-pc03's Security log (4625, 4624, 5140, 5145).\n\n## What I found\n\n- 46 failed SSH passwords from 198.51.100.23 between 01:02 and 01:19 UTC, then \"Accepted password for vendor\" at 01:19:44.\n- As root (sudo), vendor fetched /m.sh with Wget at 01:20:10 and ran it. The script fetched /a.bin with curl into /usr/local/bin/.sysmon-agent and added a root cron job every 10 minutes.\n- The program is running (PID 2644) and calls cdn.patch-relay.example over TLS every 10 minutes (01:21, 01:31, 01:41).\n- At 01:33 appt01 tried frontdesk's password on rcp-pc03 nine times, got in at 01:34:02 and read six files from the Scans share.\n\n## What I don't know yet\n\n- What .sysmon-agent does; we have not analysed the binary.\n- How the vendor account's password was guessable, and whether the vendor uses it elsewhere.\n- How frontdesk's password was known: reused, guessed or stolen.\n- Whether anything left the network apart from what the capture shows (it covers 01:15 to 01:45 only).\n\n## Disposition and why\n\nTrue positive, high: a server holding appointment data is controlled by an outsider, and patient scans were read on another machine.\n\n## Next step and owner\n\n- Clinic IT (now): take appt01 off the internet, disable vendor and frontdesk, keep a disk image before cleaning.\n- Clinic manager (today): treat the Scans access as a possible personal data breach and start the notification checks.\n- Vendor: support only through a VPN with keys, never a password on an open port.\n", "n": 6, "blank": ["text"]}
```

```tool report
{"tpl": "triage", "title": "Triage note: network finding for 6 Nov, 01:00 to 02:00 UTC", "text": "# Triage note: network finding for 6 Nov, 01:00 to 02:00 UTC\n\n## What fired\n\nZeek and the IDS, reviewed together after the appt01 alert.\n\n## What I checked\n\n`awk -F'\\t'` over zeek/conn.log for every pair (from, to, port, service), the capture with display filters, and the two IDS rules in this notebook (5 alerts, none on internal traffic).\n\n## What I found\n\n- Inbound: 198.51.100.23 made 10 SSH connections to appt01:22; the last one (01:19:44, 17 minutes, 612 kB back) is the attacker's session.\n- Outbound: appt01 opened 5 connections to 203.0.113.77: two HTTP downloads (01:20:11 /m.sh, 01:20:32 /a.bin, 2.3 MB) and three TLS beacons, 10 minutes apart, SNI cdn.patch-relay.example.\n- Internal: appt01 opened 10 SMB connections to rcp-pc03 (10.30.2.33:445); nine were refused and one lasted 157 s and returned 5.2 MB.\n- Normal: the nightly backup to 10.30.0.40 at 01:30 (412 MB, internal), database queries, the patient portal.\n\n## What I don't know yet\n\nWhat is inside the TLS beacons, and whether other servers talk to 203.0.113.77: the capture only covers the core switch for 30 minutes.\n\n## Disposition and why\n\nTrue positive: inbound brute force, outbound download and beacon, and an internal hop to a PC holding patient data.\n\n## Next step and owner\n\n- Network team: block 198.51.100.23 and 203.0.113.77 at the firewall; close inbound 22 to appt01.\n- SOC: search 30 days of Zeek logs for 203.0.113.77 and 198.51.100.23, and for any other server opening port 445 to a PC.\n", "n": 6, "blank": ["text"]}
```

- [ ] Commands and filters written so someone else can repeat them
- [ ] Times in UTC
- [ ] What you don't know is said plainly
- [ ] Each next step has an owner

> Hint: Write "we have not analysed the binary", not "it is probably ransomware". A guess in a triage note becomes a fact by the third person who reads it.

## 10. Hand in

Hand in this notebook. It carries every answer, tool result and report from the steps above.

**Rubric**

| Criterion | Points |
|---|---|
| Evidence hashed and listed before any analysis | 2 |
| Brute force and the successful login found in auth.log, with the commands used | 4 |
| What happened after the login: the download, the cron job and the running process | 4 |
| Packets: the download and the beacon found with display filters, the filters written down | 4 |
| Zeek conn.log: who talked to whom, including the connection to the reception PC | 3 |
| Windows: failed and successful logons and the share access, with event IDs | 4 |
| IDS rules that catch the download and the beacon, and nothing normal | 3 |
| Host triage note and network finding: facts kept apart from guesses, next steps with owners | 6 |

- [ ] All questions answered
- [ ] Every filter and command you used is in the notebook
- [ ] Both notes complete
- [ ] Press Hand in